[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pflahert/hd-stats-workshop/blob/main/notebooks/lab4.ipynb)

# Lab 4: Seventy Genes and a Fraud
## Penalized Regression & Biological Interpretation

**Day 2, 1:00–2:00** | Learning Outcomes: LO2, LO3

In this lab you will work through a complete penalized regression analysis: data preparation, model fitting, model selection, stability analysis, and biological interpretation. Use AI to generate code where helpful, but focus on the *decisions*: which method, which parameters, what do the results mean?

The task: **build a sparse gene signature that distinguishes ALL from AML.**

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
import warnings

# Show convergence warnings — they are pedagogically valuable in Part 1
# (we suppress other warnings to keep output clean)
warnings.filterwarnings('default', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('default')

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12

In [ ]:
# Load the Golub 1999 leukemia dataset from Zenodo
!pip install -q pyreadr
import pyreadr, urllib.request, os

url = "https://zenodo.org/records/8123245/files/leukemia_data_Golub99_3051.rda?download=1"
if not os.path.exists("golub.rda"):
    urllib.request.urlretrieve(url, "golub.rda")

rda = pyreadr.read_r("golub.rda")
expr = pd.concat([rda['golub_train_3051'], rda['golub_test_3051']]).reset_index(drop=True)
labels = pd.concat([rda['golub_train_response'].iloc[:, 0],
                     rda['golub_test_response'].iloc[:, 0]]).reset_index(drop=True)

print(f"Loaded Golub 1999 leukemia data from Zenodo.")

# expr: samples (rows) x genes (columns) — already in the right orientation
X_raw = expr.values  # 72 samples x 3,051 genes
gene_names = list(expr.columns)

# Create binary outcome: 0 = ALL, 1 = AML
y = (labels == 'AML').astype(int).values

# Standardize features (zero mean, unit variance per gene)
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

print(f"X: {X.shape[0]} samples × {X.shape[1]} genes")
print(f"y: {np.sum(y == 0)} ALL, {np.sum(y == 1)} AML")
print(f"Ratio p/n: {X.shape[1] / X.shape[0]:.1f}")

---
## Part 1: Why OLS Fails

Before we add any penalty, let's try to fit logistic regression without regularization on all 3,051 genes. Watch it fail or overfit.

In [ ]:
np.random.seed(2026)

# Unregularized logistic regression — no penalty
lr_none = LogisticRegression(penalty=None, max_iter=10000, random_state=2026)
lr_none.fit(X, y)

# Training accuracy
train_acc = lr_none.score(X, y)

# Cross-validation accuracy
cv_scores = cross_val_score(lr_none, X, y, cv=10, scoring='accuracy')

print("Unregularized logistic regression (no penalty):")
print(f"  Training accuracy:      {train_acc:.4f}")
print(f"  CV accuracy (10-fold):  {cv_scores.mean():.4f} \u00b1 {cv_scores.std():.4f}")
print()
print("Training accuracy is perfect (or near-perfect), but CV accuracy is much lower.")
print("This is the overfitting catastrophe: with p >> n, the model memorizes the training data.")

---
## Part 2: Ridge Regression

Ridge regression adds an L2 penalty: it shrinks all coefficients toward zero but keeps all genes in the model. No gene is ever exactly zeroed out.

In [ ]:
np.random.seed(2026)

# Ridge logistic regression with 10-fold CV
ridge = LogisticRegressionCV(
    penalty='l2',
    Cs=20,
    cv=10,
    max_iter=10000,
    random_state=2026,
    scoring='accuracy'
)
ridge.fit(X, y)

# CV accuracy curve
# scores_ is a dict keyed by class label; for binary, use the positive class
pos_class = ridge.classes_[1]
cs = ridge.Cs_
mean_scores = ridge.scores_[pos_class].mean(axis=0)
std_scores = ridge.scores_[pos_class].std(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: CV accuracy vs C
axes[0].semilogx(cs, mean_scores, 'o-', color='#4C72B0')
axes[0].fill_between(cs, mean_scores - std_scores, mean_scores + std_scores, alpha=0.2)
axes[0].axvline(x=ridge.C_[0], color='red', linestyle='--', label=f'Best C = {ridge.C_[0]:.4f}')
axes[0].set_xlabel('C (inverse regularization strength)')
axes[0].set_ylabel('CV Accuracy')
axes[0].set_title('Ridge: CV Accuracy vs C')
axes[0].legend()

# Right: histogram of coefficient values
ridge_coefs = ridge.coef_.ravel()
axes[1].hist(ridge_coefs, bins=80, edgecolor='white', color='#4C72B0', alpha=0.8)
axes[1].set_xlabel('Coefficient value')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Ridge: Distribution of coefficients')

plt.tight_layout()
plt.show()

best_idx = np.argmax(mean_scores)
print(f"Best C: {ridge.C_[0]:.4f} (recall: C = 1/\u03bb, so smaller C = stronger penalty)")
print(f"Best CV accuracy: {mean_scores[best_idx]:.4f}")
print(f"Coefficients exactly zero: {np.sum(ridge_coefs == 0)}")
print(f"Coefficients |coef| > 0.01: {np.sum(np.abs(ridge_coefs) > 0.01)}")
print(f"Total coefficients: {len(ridge_coefs)}")

### Critique checklist

| Question | Your answer |
|----------|-------------|
| Did it use `penalty='l2'`? | |
| How many coefficients are exactly zero? (Answer: none — that's ridge) | |
| What is the CV accuracy? | |

---
## Part 3: LASSO

LASSO adds an L1 penalty, which gives sparse solutions — some coefficients become exactly zero. This performs variable selection: the genes that survive are the LASSO's chosen signature.

In [ ]:
np.random.seed(2026)

# LASSO logistic regression with 10-fold CV
lasso = LogisticRegressionCV(
    penalty='l1',
    solver='saga',
    Cs=20,
    cv=10,
    max_iter=10000,
    random_state=2026,
    scoring='accuracy'
)
lasso.fit(X, y)

# CV accuracy curve
pos_class = lasso.classes_[1]
cs_l1 = lasso.Cs_
mean_scores_l1 = lasso.scores_[pos_class].mean(axis=0)
std_scores_l1 = lasso.scores_[pos_class].std(axis=0)

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(cs_l1, mean_scores_l1, 'o-', color='#DD8452')
ax.fill_between(cs_l1, mean_scores_l1 - std_scores_l1, mean_scores_l1 + std_scores_l1, alpha=0.2, color='#DD8452')
ax.axvline(x=lasso.C_[0], color='red', linestyle='--', label=f'Best C = {lasso.C_[0]:.4f}')
ax.set_xlabel('C (inverse regularization strength)')
ax.set_ylabel('CV Accuracy')
ax.set_title('LASSO: CV Accuracy vs C')
ax.legend()
plt.tight_layout()
plt.show()

# Extract selected genes
lasso_coefs = lasso.coef_.ravel()
nonzero_mask = lasso_coefs != 0
selected_genes = [gene_names[i] for i in range(len(gene_names)) if nonzero_mask[i]]
selected_coefs = lasso_coefs[nonzero_mask]

best_idx_l1 = np.argmax(mean_scores_l1)
print(f"Best C: {lasso.C_[0]:.4f}")
print(f"Best CV accuracy: {mean_scores_l1[best_idx_l1]:.4f}")
print(f"Genes selected (nonzero coefficients): {len(selected_genes)}")
print(f"\nSelected genes:")
for g, c in sorted(zip(selected_genes, selected_coefs), key=lambda x: abs(x[1]), reverse=True):
    direction = "AML marker" if c > 0 else "ALL marker"
    print(f"  {g:>15s}: {c:+.4f}  ({direction})")

In [ ]:
# Bar chart of LASSO coefficients, colored by direction
order = np.argsort(np.abs(selected_coefs))[::-1]
ordered_genes = [selected_genes[i] for i in order]
ordered_coefs = selected_coefs[order]
colors = ['#DD8452' if c > 0 else '#4C72B0' for c in ordered_coefs]

fig, ax = plt.subplots(figsize=(10, max(4, len(ordered_genes) * 0.35)))
ax.barh(range(len(ordered_genes)), ordered_coefs, color=colors, edgecolor='white')
ax.set_yticks(range(len(ordered_genes)))
ax.set_yticklabels(ordered_genes, fontsize=10)
ax.set_xlabel('LASSO Coefficient')
ax.set_title('LASSO: Nonzero Coefficients (Orange = AML, Blue = ALL)')
ax.invert_yaxis()
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

### Exercises

**3.1** How many genes were selected by LASSO?

**YOUR ANSWER:**

**3.2** Would you report the model with the best C, or a more parsimonious one? Why? (Connect this to the $\lambda_{\min}$ vs. $\lambda_{1\text{se}}$ concept from the lecture.)

**YOUR ANSWER:**

---
## Part 4: Elastic Net

Elastic net combines L1 and L2 penalties — a compromise that handles correlated predictors better than LASSO alone. When genes are co-expressed, LASSO tends to pick one and drop the rest; elastic net keeps groups of correlated genes together.

In [ ]:
np.random.seed(2026)

# Elastic net with l1_ratio = 0.5 (equal mix of L1 and L2)
enet = LogisticRegressionCV(
    penalty='elasticnet',
    solver='saga',
    l1_ratios=[0.5],
    Cs=20,
    cv=10,
    max_iter=10000,
    random_state=2026,
    scoring='accuracy'
)
enet.fit(X, y)

# Extract elastic net selected genes
enet_coefs = enet.coef_.ravel()
enet_nonzero = enet_coefs != 0
enet_genes = set(gene_names[i] for i in range(len(gene_names)) if enet_nonzero[i])
lasso_genes_set = set(selected_genes)

# Overlap analysis
overlap = lasso_genes_set & enet_genes
lasso_only = lasso_genes_set - enet_genes
enet_only = enet_genes - lasso_genes_set

# scores_[pos_class] for elastic net has shape (n_folds, n_Cs, n_l1_ratios) — ravel to 1-d
pos_class = enet.classes_[1]
mean_scores_enet = enet.scores_[pos_class].mean(axis=0).ravel()
best_enet_idx = int(np.argmax(mean_scores_enet))

print(f"Elastic net best C: {enet.C_[0]:.4f}")
print(f"Elastic net CV accuracy: {mean_scores_enet[best_enet_idx]:.4f}")
print(f"Elastic net genes selected: {len(enet_genes)}")
print()
print(f"Overlap analysis:")
print(f"  Both LASSO and elastic net: {len(overlap)}")
print(f"  LASSO only:                 {len(lasso_only)}")
print(f"  Elastic net only:           {len(enet_only)}")

if lasso_only:
    print(f"\nLASSO-only genes: {sorted(lasso_only)}")
if enet_only:
    print(f"Elastic-net-only genes: {sorted(enet_only)}")

In [ ]:
# Compare coefficient magnitudes for shared genes
if overlap:
    shared_genes_list = sorted(overlap)
    gene_to_idx = {g: i for i, g in enumerate(gene_names)}
    lasso_shared = [lasso_coefs[gene_to_idx[g]] for g in shared_genes_list]
    enet_shared = [enet_coefs[gene_to_idx[g]] for g in shared_genes_list]

    fig, ax = plt.subplots(figsize=(8, 5))
    x_pos = np.arange(len(shared_genes_list))
    width = 0.35
    ax.bar(x_pos - width/2, lasso_shared, width, label='LASSO', color='#DD8452', edgecolor='white')
    ax.bar(x_pos + width/2, enet_shared, width, label='Elastic Net', color='#55A868', edgecolor='white')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(shared_genes_list, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Coefficient')
    ax.set_title('Shared Genes: LASSO vs Elastic Net Coefficients')
    ax.legend()
    ax.axhline(y=0, color='black', linewidth=0.5)
    plt.tight_layout()
    plt.show()

### Exercise

**4.1** Does elastic net select more or fewer genes than LASSO? Explain why, based on the properties of the L1 + L2 penalty.

**YOUR ANSWER:**

---
## Part 5: Stability Analysis

LASSO selection is unstable: small changes in the data can flip genes in or out of the model. Bootstrap resampling reveals which genes are *consistently* selected. These are the ones you should trust.

In [ ]:
np.random.seed(2026)

# Bootstrap stability: liblinear is much faster than saga for L1 logistic at this scale,
# and 25 fits is enough to estimate selection frequencies for a stability plot.
n_boot = 25
n_samples = X.shape[0]
n_genes = X.shape[1]
selection_counts = np.zeros(n_genes)

best_C = lasso.C_[0]

print(f"Running {n_boot} bootstrap LASSO fits (C = {best_C:.4f})...")
for b in range(n_boot):
    # Bootstrap sample (with replacement)
    idx = np.random.choice(n_samples, size=n_samples, replace=True)
    X_boot = X[idx]
    y_boot = y[idx]

    # Fit LASSO with the best C from earlier; liblinear is fast for L1 logistic
    lr_boot = LogisticRegression(
        penalty='l1',
        solver='liblinear',
        C=best_C,
        max_iter=10000,
        random_state=2026
    )
    lr_boot.fit(X_boot, y_boot)

    # Record nonzero coefficients
    selection_counts += (lr_boot.coef_.ravel() != 0).astype(int)

    if (b + 1) % 5 == 0:
        print(f"  Completed {b + 1}/{n_boot}")

# Selection frequency
selection_freq = selection_counts / n_boot

# Top 20 genes by selection frequency
top_idx = np.argsort(selection_freq)[::-1][:20]
top_genes = [gene_names[i] for i in top_idx]
top_freqs = selection_freq[top_idx]

fig, ax = plt.subplots(figsize=(10, 7))
colors_boot = ['#DD8452' if f >= 0.5 else '#999999' for f in top_freqs]
ax.barh(range(len(top_genes)), top_freqs, color=colors_boot, edgecolor='white')
ax.set_yticks(range(len(top_genes)))
ax.set_yticklabels(top_genes, fontsize=10)
ax.set_xlabel('Selection Frequency')
ax.set_title('Bootstrap Stability: Top 20 Genes by Selection Frequency')
ax.axvline(x=0.5, color='red', linestyle='--', linewidth=1.5, label='50% threshold')
ax.set_xlim(0, 1.05)
ax.invert_yaxis()
ax.legend()
plt.tight_layout()
plt.show()

# Summary counts
n_50 = np.sum(selection_freq > 0.5)
n_80 = np.sum(selection_freq > 0.8)
n_90 = np.sum(selection_freq > 0.9)
print(f"Genes selected in >50% of bootstraps: {n_50}")
print(f"Genes selected in >80% of bootstraps: {n_80}")
print(f"Genes selected in >90% of bootstraps: {n_90}")

### Critique checklist

| Question | Your answer |
|----------|-------------|
| Did bootstrapping use `replace=True`? | |
| How many genes selected in >50% of bootstraps? | |
| Are the most stable genes the same as the largest LASSO coefficients? | |

### Exercise

**5.1** A colleague says "LASSO selected 15 genes, so these cause the disease." What's wrong with this? Use the bootstrap results to support your answer.

**YOUR ANSWER:**

---
## Part 6: Model Comparison

Compare all three methods side by side. Same data, same cross-validation, different penalties.

In [ ]:
np.random.seed(2026)

# Compare using the internal CV scores from each fitted model
# Note: these C values were already tuned on the full data via LogisticRegressionCV,
# so the CV accuracy is slightly optimistic. For a rigorous comparison you would
# need nested CV (outer CV for evaluation, inner CV for tuning). We accept this
# limitation here for simplicity.

# Use internal CV scores at the best C; ravel to handle the elastic-net 2-d case uniformly
pos_class_ridge = ridge.classes_[1]
pos_class_lasso = lasso.classes_[1]
pos_class_enet = enet.classes_[1]

ridge_mean = ridge.scores_[pos_class_ridge].mean(axis=0).ravel()
lasso_mean = lasso.scores_[pos_class_lasso].mean(axis=0).ravel()
enet_mean = enet.scores_[pos_class_enet].mean(axis=0).ravel()

ridge_best_cv = float(ridge_mean[np.argmax(ridge_mean)])
lasso_best_cv = float(lasso_mean[np.argmax(lasso_mean)])
enet_best_cv = float(enet_mean[np.argmax(enet_mean)])

# Count nonzero coefficients
n_nonzero_ridge = int(np.sum(ridge.coef_.ravel() != 0))
n_nonzero_lasso = int(np.sum(lasso.coef_.ravel() != 0))
n_nonzero_enet = int(np.sum(enet.coef_.ravel() != 0))

comparison = pd.DataFrame({
    'Method': ['Ridge (L2)', 'LASSO (L1)', 'Elastic Net (L1+L2)'],
    'CV Accuracy': [ridge_best_cv, lasso_best_cv, enet_best_cv],
    'Nonzero Genes': [n_nonzero_ridge, n_nonzero_lasso, n_nonzero_enet]
})

print(comparison.to_string(index=False))
print("\nNote: CV accuracy comes from LogisticRegressionCV's internal cross-validation.")
print("The C hyperparameter was tuned on the same data, so these are slightly optimistic.")

### Exercise

**6.1** If all three methods have similar accuracy but very different gene counts, what does this tell you about the data?

**YOUR ANSWER:**

---
## Part 7: Heatmap of Stable Genes

Visualize the stably selected genes across all 72 samples. If LASSO found a real signature, these genes should visually separate ALL from AML samples.

In [ ]:
# Get genes selected in >50% of bootstraps
stable_idx = np.where(selection_freq > 0.5)[0]
stable_genes = [gene_names[i] for i in stable_idx]

if len(stable_genes) == 0:
    print("No genes above 50% threshold. Lowering to 30%.")
    stable_idx = np.where(selection_freq > 0.3)[0]
    stable_genes = [gene_names[i] for i in stable_idx]

print(f"Stable genes for heatmap: {len(stable_genes)}")

# Extract expression values (samples x selected genes), re-standardized
sample_labels = [f"S{i}" for i in range(X.shape[0])]
heatmap_data = pd.DataFrame(
    X[:, stable_idx],
    columns=stable_genes,
    index=sample_labels
).T

# Color annotation for ALL/AML subtype
subtype_labels = labels.values
subtype_map = {'ALL': '#4C72B0', 'AML': '#DD8452'}
col_colors = [subtype_map[s] for s in subtype_labels]

g = sns.clustermap(
    heatmap_data,
    cmap='RdBu_r',
    center=0,
    col_colors=col_colors,
    figsize=(14, max(6, len(stable_genes) * 0.4)),
    dendrogram_ratio=(0.1, 0.15),
    cbar_pos=(0.02, 0.8, 0.03, 0.15),
    yticklabels=True,
    xticklabels=False
)
g.ax_heatmap.set_xlabel('Samples (Blue = ALL, Orange = AML)')
g.ax_heatmap.set_ylabel('Genes')
g.fig.suptitle('Stably Selected Genes Across Leukemia Samples', y=1.02, fontsize=14)
plt.show()

---
## Part 8: AI Extension

Ask an AI to compare your gene signature to known leukemia biology.

**Suggested prompt:**

> "These genes were identified as stably distinguishing ALL from AML in a LASSO analysis with bootstrap stability: [paste gene list]. For each gene, briefly describe its known function and relevance to ALL/AML biology or leukemia."

Copy the stable gene list from above and paste it into your AI assistant.

In [ ]:
# Print stable gene list for easy copy-paste to AI
print("Stable gene list (>50% bootstrap frequency):")
print(", ".join(stable_genes))

In [ ]:
# YOUR CODE HERE (AI-generated or your own)


### Verification Table

Verify at least 3 genes using [GeneCards](https://www.genecards.org/) or [NCBI Gene](https://www.ncbi.nlm.nih.gov/gene/).

| Gene | AI's claim | What GeneCards/NCBI says | Accurate? |
|------|-----------|------------------------|----------|
| | | | |
| | | | |
| | | | |

**Did the AI hallucinate any gene functions?**

**YOUR ANSWER:**

---
## Part 9: Connecting Day 1 and Day 2

**9.1** How could you combine FDR analysis (Day 1) with LASSO (today) in a single pipeline?

**YOUR ANSWER:**

**9.2** Would you expect the BH-FDR gene list and LASSO gene list to overlap completely? Why or why not?

(Hint: FDR finds differential expression; LASSO finds prediction. Redundant genes may be FDR-significant but dropped by LASSO because they add no predictive value beyond what correlated genes already provide.)

**YOUR ANSWER:**

---
## Wrap-Up Questions

1. When would you choose ridge over LASSO in a genomics application?

   **YOUR ANSWER:**

2. "LASSO selected 15 genes, so these cause the disease." What's wrong with this interpretation?

   **YOUR ANSWER:**

3. How could you combine FDR analysis and LASSO in a single pipeline?

   **YOUR ANSWER:**

4. What did you learn verifying AI's gene descriptions against GeneCards?

   **YOUR ANSWER:**